In [30]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))

In [31]:
from src.data.filter import load_data, filter_with_nation_winners
from src.data.preprocess import Dataset

In [32]:
data = load_data()
data = filter_with_nation_winners(data)

dataset = Dataset(data)
first_innings_data, second_innings_data = dataset.process_data(insert_team_indices=True)

In [33]:
import pandas as pd

df = pd.DataFrame(second_innings_data)
df.head()

,runs,wickets,balls,labels,batting_team,bowling_team
0,283,10,299,1,14,19
1,283,10,298,1,14,19
2,283,10,297,1,14,19
3,283,10,296,1,14,19
4,279,10,295,1,14,19


In [39]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer

X = df[["batting_team", "bowling_team", "runs", "wickets", "balls"]]
y = df["labels"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", 
         OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas"), 
         ["batting_team", "bowling_team"]),
        ("num", MinMaxScaler(), ["runs", "wickets", "balls"])
    ],
    remainder="passthrough"
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)
print(X_train[0])

len(X_train), len(X_test), len(y_train), len(y_test)

[0.         1.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         1.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.48015873 1.         0.99666667]


(419832, 104958, 419832, 104958)

In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

model = LogisticRegression(max_iter=1500)
model.fit(X_train, y_train)
log_loss(y_test, model.predict_proba(X_test)), log_loss(y_train, model.predict_proba(X_train))


(0.35725915156696414, 0.35474321071121445)